# QSVM Experiment 5: the {raw, PCA} x {mean, mean+std} ablationSteve's Section 7, item 5: run the complete two by two comparison{raw, PCA} x {mean, mean plus standard deviation}, to check whether the earlierdeterioration seen between the PCA pooled representation and the raw mean plusstandard deviation representation is about PCA specifically, or about dimensionalityin general.**Why this is a separate notebook and not a reuse of the old PCA dataset.** Theoriginal PCA-pooled file (PCA_scaled_dataframe.csv / PCA_4_full_dataset_36.csv) wasbuilt from an outdated version of the z-scale lookup table, the same discrepancy foundearlier between zscale_dataset.csv and moments_order1_z5.csv. Rebuilding it here fromzscaled_5_45.csv keeps everything on the current, consistent table.**A structural note on the PCA side.** The original manuscript's PCA reduces eachposition separately (5 z-values per position down to 4), keeping the 9-position axisintact until the final pooling step. The PCA built in Experiment 2 works differently:it reduces the full 45 concatenated features (all positions and z-values mixedtogether) directly to a target size, so no separate position axis survives to poolafterwards. To still build the 2x2 using that same method, "mean" versus "mean+std" onthe PCA side is read here as two levels of detail of the same fold-safe PCA rather thantwo pooling operators: PCA to 5 components (matching mean_z's 5 dimensions) and PCA to10 components (matching the extra dimensionality that mean+std adds on the raw side,5 to 10). This keeps both sides of the 2x2 dimensionally matched.**Scope.** Classical kernels only, consistent with the rest of the Step-4 diagnostics:even though all four representations here are 10 dimensions or fewer (technically withinthe quantum feasibility ceiling used earlier), the same efficiency reasoning applies:Step 3 already showed quantum and classical kernels agreeing within their paireduncertainty intervals, so re-adding quantum here would mostly cost time without changingthe conclusion. Flag it if you want the quantum kernels added back for this specificablation.|            | mean (5 dims)  | mean+std (10 dims) ||------------|----------------|--------------------|| **raw**    | `raw_mean` = mean_z | `raw_mean_std` = moments order 2 || **PCA**    | `pca_mean` = PCA(45->5), fold-safe | `pca_mean_std` = PCA(45->10), fold-safe |

## 1. Configuration and imports

In [1]:
import os
import time
import itertools
import numpy as np
import pandas as pd
from scipy import stats
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, accuracy_score, matthews_corrcoef
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, polynomial_kernel

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_SPLITS_OUTER, N_REPEATS_OUTER, N_SPLITS_INNER = 5, 20, 4
CLASSICAL_GRIDS = {
    'linear': {'C': [0.1, 1, 10]},
    'rbf':    {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1, 1]},
    'poly':   {'C': [0.1, 1, 10], 'degree': [2, 3], 'gamma': ['scale', 0.1, 1], 'coef0': [0.0, 1.0]},
}
MODELS = ['linear', 'rbf', 'poly']

OUTDIR = 'results_experiment5'
os.makedirs(OUTDIR, exist_ok=True)
print(f'outer {N_SPLITS_OUTER}x{N_REPEATS_OUTER} = {N_SPLITS_OUTER*N_REPEATS_OUTER} folds, inner {N_SPLITS_INNER}')

outer 5x20 = 100 folds, inner 4


## 2. Data, sequences, groups

In [2]:
_lab = pd.read_excel('Docking_high_low_energy_labels.xlsx', sheet_name='Results')
_lab = _lab.dropna(subset=['Sequence Epitope)']).reset_index(drop=True)
sequences = _lab['Sequence Epitope)'].astype(str).str.strip().tolist()
y = (_lab['Otsu Class theshold -77.8'] == 'Strong').astype(int).values

full45 = pd.read_csv('zscaled_5_45.csv', index_col=0).values  # current, up-to-date z-table
assert full45.shape == (len(y), 45)
print(f'{len(y)} peptides, class balance {np.bincount(y)}')

  warn("""Cannot parse header or footer so it will be ignored""")
80 peptides, class balance [42 38]


In [3]:
K_SHARED, IDENTITY_THRESH = 5, 7 / 9


def similarity_edges(sequences, k_shared=K_SHARED, identity_thresh=IDENTITY_THRESH):
    n, L = len(sequences), len(sequences[0])
    kmers = [{s[i:i + k_shared] for i in range(L - k_shared + 1)} for s in sequences]
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            shared = bool(kmers[i] & kmers[j])
            identity = sum(a == b for a, b in zip(sequences[i], sequences[j])) / L
            if shared or identity >= identity_thresh:
                edges.append((i, j))
    return edges


def build_groups_unionfind(n, edges):
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for i, j in edges:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj
    raw = [find(i) for i in range(n)]
    remap = {g: k for k, g in enumerate(sorted(set(raw)))}
    return np.array([remap[g] for g in raw])


n_pep = len(sequences)
edges = similarity_edges(sequences)
groups = build_groups_unionfind(n_pep, edges)

adj = coo_matrix((np.ones(len(edges)),
                  (np.array([e[0] for e in edges]), np.array([e[1] for e in edges]))),
                 shape=(n_pep, n_pep))
n_cc, cc_labels = connected_components(adj, directed=False)
assert len(set(zip(groups, cc_labels))) == len(set(groups)) == len(set(cc_labels))
print(f'{len(np.unique(groups))} groups (verified against scipy), '
      f'{pd.Series(groups).value_counts().eq(1).sum()} singletons')

43 groups (verified against scipy), 25 singletons


## 3. The four representations of the 2x2

In [4]:
def mean_from_full45(X45, exclude_positions=()):
    keep = [p for p in range(9) if p not in exclude_positions]
    out = np.zeros((X45.shape[0], 5))
    for k in range(5):
        cols = [p * 5 + k for p in keep]
        out[:, k] = X45[:, cols].mean(axis=1)
    return out


def std_from_full45(X45):
    out = np.zeros((X45.shape[0], 5))
    for k in range(5):
        cols = [p * 5 + k for p in range(9)]
        out[:, k] = X45[:, cols].std(axis=1)
    return out


raw_mean = mean_from_full45(full45)                              # 5 dims
raw_mean_std = np.hstack([raw_mean, std_from_full45(full45)])     # 10 dims

print('raw_mean     ', raw_mean.shape)
print('raw_mean_std ', raw_mean_std.shape)


def fit_scale(X_train, X_test):
    scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
    scaler.fit(X_train)
    return scaler.transform(X_train), scaler.transform(X_test)


def transform_pca(X_train, X_test, n_components):
    scaler = StandardScaler().fit(X_train)
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED).fit(scaler.transform(X_train))
    return pca.transform(scaler.transform(X_train)), pca.transform(scaler.transform(X_test))


def _transform(rep, tr_idx, te_idx):
    if callable(rep):
        return rep(tr_idx, te_idx)
    return fit_scale(rep[tr_idx], rep[te_idx])


REPS = {
    'raw_mean': raw_mean,
    'raw_mean_std': raw_mean_std,
    'pca_mean': lambda tr, te: transform_pca(full45[tr], full45[te], n_components=5),
    'pca_mean_std': lambda tr, te: transform_pca(full45[tr], full45[te], n_components=10),
}

raw_mean      (80, 5)
raw_mean_std  (80, 10)


## 4. Nested-CV machinery (same discipline as the other notebooks)

In [5]:
def make_outer_splits(y, groups=None, n_splits=N_SPLITS_OUTER, n_repeats=N_REPEATS_OUTER,
                      random_state=RANDOM_SEED):
    if groups is None:
        return list(RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                            random_state=random_state).split(np.zeros(len(y)), y))
    splits = []
    for r in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state + r)
        splits.extend(sgkf.split(np.zeros(len(y)), y, groups))
    return splits


def _inner_splits(y_tr, groups_tr, fold_i):
    if groups_tr is None:
        return list(StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True,
                                    random_state=RANDOM_SEED + fold_i).split(np.zeros(len(y_tr)), y_tr))
    return list(StratifiedGroupKFold(n_splits=N_SPLITS_INNER, shuffle=True,
                                     random_state=RANDOM_SEED + fold_i)
                .split(np.zeros(len(y_tr)), y_tr, groups_tr))


def _param_combinations(grid):
    keys = list(grid)
    return [dict(zip(keys, vals)) for vals in itertools.product(*(grid[k] for k in keys))]


def _resolve_gamma(gamma, Xs):
    if gamma == 'scale':
        return 1.0 / (Xs.shape[1] * Xs.var()) if Xs.var() > 0 else 1.0
    return gamma


def _fit_svc(kernel_name, Xtr, ytr, params):
    gamma = _resolve_gamma(params.get('gamma', 'scale'), Xtr)
    if kernel_name == 'linear':
        clf = SVC(kernel='linear', C=params['C'])
    elif kernel_name == 'rbf':
        clf = SVC(kernel='rbf', C=params['C'], gamma=gamma)
    else:
        clf = SVC(kernel='poly', C=params['C'], degree=params['degree'], gamma=gamma, coef0=params['coef0'])
    return clf.fit(Xtr, ytr)


def nested_cv_classical(rep, y, kernel_name, groups=None, n_repeats_outer=N_REPEATS_OUTER):
    combos = _param_combinations(CLASSICAL_GRIDS[kernel_name])
    outer = make_outer_splits(y, groups, n_repeats=n_repeats_outer)
    rows, predictions = [], {}

    for fold_i, (tr_idx, te_idx) in enumerate(outer):
        y_tr, y_te = y[tr_idx], y[te_idx]
        inner = _inner_splits(y_tr, None if groups is None else groups[tr_idx], fold_i)

        inner_scores = {}
        for in_tr, in_val in inner:
            abs_tr, abs_val = tr_idx[in_tr], tr_idx[in_val]
            Xi_tr, Xi_val = _transform(rep, abs_tr, abs_val)
            for ci, params in enumerate(combos):
                clf = _fit_svc(kernel_name, Xi_tr, y[abs_tr], params)
                s = balanced_accuracy_score(y[abs_val], clf.predict(Xi_val))
                inner_scores.setdefault(ci, []).append(s)

        best_ci = max(inner_scores, key=lambda k: np.mean(inner_scores[k]))
        best_params = combos[best_ci]

        Xtr, Xte = _transform(rep, tr_idx, te_idx)
        clf = _fit_svc(kernel_name, Xtr, y_tr, best_params)
        y_pred = clf.predict(Xte)

        row = {'fold': fold_i, 'inner_score': float(np.mean(inner_scores[best_ci])),
               'accuracy': accuracy_score(y_te, y_pred),
               'balanced_accuracy': balanced_accuracy_score(y_te, y_pred),
               'mcc': matthews_corrcoef(y_te, y_pred) if len(set(y_pred)) > 1 else 0.0,
               'n_train': len(tr_idx), 'n_test': len(te_idx)}
        row.update({f'best_{k}': v for k, v in best_params.items()})
        rows.append(row)
        predictions[fold_i] = {'test_idx': te_idx.tolist(), 'y_true': y_te.tolist(), 'y_pred': y_pred.tolist()}
    return pd.DataFrame(rows), predictions


def run_representation(name, rep, models=MODELS, groups=groups, n_repeats_outer=N_REPEATS_OUTER):
    return {kn: nested_cv_classical(rep, y, kn, groups=groups, n_repeats_outer=n_repeats_outer)
           for kn in models}


def summarize(results_by_rep):
    rows = []
    for rep_name, by_model in results_by_rep.items():
        for kn, (res, preds) in by_model.items():
            rows.append({'representation': rep_name, 'model': kn,
                         'balanced_accuracy': res['balanced_accuracy'].mean()})
    return pd.DataFrame(rows).pivot(index='representation', columns='model', values='balanced_accuracy')

In [6]:
def per_fold_metrics(predictions):
    accs, baccs, mccs = [], [], []
    for fold_i in sorted(predictions, key=int):
        yt = np.asarray(predictions[fold_i]['y_true'])
        yp = np.asarray(predictions[fold_i]['y_pred'])
        accs.append(accuracy_score(yt, yp))
        baccs.append(balanced_accuracy_score(yt, yp))
        mccs.append(matthews_corrcoef(yt, yp) if len(set(yp)) > 1 else 0.0)
    return {'accuracy': np.array(accs), 'balanced_accuracy': np.array(baccs), 'mcc': np.array(mccs)}


def nadeau_bengio_ci(diff, n_train, n_test, alpha=0.05):
    diff = np.asarray(diff, dtype=float)
    J = len(diff)
    var = diff.var(ddof=1)
    se = np.sqrt(var * (1.0 / J + n_test / n_train))
    t = stats.t.ppf(1 - alpha / 2, df=J - 1)
    mean = diff.mean()
    t_stat = mean / se if se > 0 else np.nan
    p = 2 * (1 - stats.t.cdf(abs(t_stat), df=J - 1)) if np.isfinite(t_stat) else np.nan
    return float(mean), float(mean - t * se), float(mean + t * se), float(p)


def _peptide_tallies(predictions, n_samples):
    n_pred, n_correct = np.zeros(n_samples), np.zeros(n_samples)
    for fold_i in sorted(predictions, key=int):
        idx = np.asarray(predictions[fold_i]['test_idx'])
        yt, yp = np.asarray(predictions[fold_i]['y_true']), np.asarray(predictions[fold_i]['y_pred'])
        n_pred[idx] += 1
        n_correct[idx] += (yt == yp)
    return n_pred, n_correct


def bootstrap_group_ci(pred_a, pred_b, groups, y, n_boot=10000, seed=RANDOM_SEED):
    n = len(y)
    na, ca = _peptide_tallies(pred_a, n)
    nb, cb = _peptide_tallies(pred_b, n)
    uniq = np.unique(groups)
    members = [np.where(groups == g)[0] for g in uniq]
    pos = (y == 1)

    def bacc(n_pred, n_corr, idx):
        p, q = idx[pos[idx]], idx[~pos[idx]]
        tpr = n_corr[p].sum() / n_pred[p].sum() if n_pred[p].sum() > 0 else np.nan
        tnr = n_corr[q].sum() / n_pred[q].sum() if n_pred[q].sum() > 0 else np.nan
        return 0.5 * (tpr + tnr)

    rng = np.random.default_rng(seed)
    all_idx = np.arange(n)
    observed = bacc(na, ca, all_idx) - bacc(nb, cb, all_idx)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), size=len(uniq))])
        boots[b] = bacc(na, ca, idx) - bacc(nb, cb, idx)
    boots = boots[np.isfinite(boots)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(observed), float(lo), float(hi)


def paired_report(res_a, pred_a, res_b, pred_b, groups, y, label_a, label_b):
    ma, mb = per_fold_metrics(pred_a), per_fold_metrics(pred_b)
    d = ma['balanced_accuracy'] - mb['balanced_accuracy']
    n_tr, n_te = res_a['n_train'].mean(), res_a['n_test'].mean()
    m_n, lo_n, hi_n, p_n = nadeau_bengio_ci(d, n_tr, n_te)
    m_g, lo_g, hi_g = bootstrap_group_ci(pred_a, pred_b, groups, y)
    return {'comparison': f'{label_a} vs {label_b}',
            f'BA_{label_a}': ma['balanced_accuracy'].mean(), f'BA_{label_b}': mb['balanced_accuracy'].mean(),
            'mean_diff': m_n, 'nadeau_bengio_lo': lo_n, 'nadeau_bengio_hi': hi_n,
            'nadeau_bengio_excl_0': (lo_n > 0) or (hi_n < 0),
            'group_boot_diff': m_g, 'group_boot_lo': lo_g, 'group_boot_hi': hi_g,
            'group_boot_excl_0': (lo_g > 0) or (hi_g < 0)}

## 5. Run the 2x2

In [7]:
t0 = time.time()
res = {name: run_representation(name, rep) for name, rep in REPS.items()}
print(f'total: {time.time()-t0:.0f}s')

tab = summarize(res)
tab = tab.loc[['raw_mean', 'raw_mean_std', 'pca_mean', 'pca_mean_std']]
tab.to_csv(f'{OUTDIR}/exp5_summary.csv')
display(tab)

total: 157s
model             linear      poly       rbf
representation                              
raw_mean        0.831438  0.839841  0.819970
raw_mean_std    0.777004  0.782659  0.778234
pca_mean        0.614048  0.571171  0.592490
pca_mean_std    0.629881  0.637599  0.632748


## 6. All six pairwise comparisons

In [8]:
PAIRS = [('raw_mean', 'raw_mean_std'), ('raw_mean', 'pca_mean'), ('raw_mean', 'pca_mean_std'),
        ('raw_mean_std', 'pca_mean'), ('raw_mean_std', 'pca_mean_std'), ('pca_mean', 'pca_mean_std')]

comp = pd.DataFrame(sum([[
    paired_report(*res[a][kn], *res[b][kn], groups, y, f'{a}[{kn}]', f'{b}[{kn}]')
    for a, b in PAIRS
] for kn in MODELS], [])).set_index('comparison')
comp.to_csv(f'{OUTDIR}/exp5_paired.csv')
display(comp[['mean_diff', 'nadeau_bengio_lo', 'nadeau_bengio_hi', 'nadeau_bengio_excl_0',
              'group_boot_lo', 'group_boot_hi', 'group_boot_excl_0']])

                                              mean_diff  nadeau_bengio_lo  nadeau_bengio_hi  nadeau_bengio_excl_0  group_boot_lo  group_boot_hi  group_boot_excl_0
comparison                                                                                                                                                        
raw_mean[linear] vs raw_mean_std[linear]       0.054435         -0.030550          0.139420                 False      -0.004104       0.129892              False
raw_mean[linear] vs pca_mean[linear]           0.217391          0.085461          0.349320                  True       0.117981       0.301724               True
raw_mean[linear] vs pca_mean_std[linear]       0.201558          0.077904          0.325211                  True       0.115205       0.274814               True
raw_mean_std[linear] vs pca_mean[linear]       0.162956          0.012949          0.312964                  True       0.039208       0.269345               True
raw_mean_std[linear] v

## 7. Reading thisThe comparison Steve specifically flagged was the deterioration between the PCA-pooledrepresentation and the raw mean+std representation (`raw_mean_std` vs `pca_mean_std`here, matching dimensionality at 10 features each). If that comparison excludes zerowith PCA behind, the deterioration is real and specific to PCA, not just an artefact ofwhich features happen to be included. The `raw_mean` vs `pca_mean` row is the samequestion at the lower (5-feature) dimensionality. The remaining four rows (mixingdimensionalities) help separate a pure PCA effect from a pure dimensionality effect,the same logic as Experiment 2's dimension-matched controls.